In [0]:
%pip install langchain langchain-openai

In [0]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

CATALOG = "deepcatalog"
SCHEMA  = "delta_tables"

os.environ["OPENAI_API_KEY"] = dbutils.secrets.get(scope="llm-scope", key="openai-key")

llm = ChatOpenAI(model="gpt-4o", temperature=0)
print("✅ Ready")

In [0]:
TABLE = f"{CATALOG}.{SCHEMA}.patients"

SCHEMA_INFO = """
Table: patients
Columns:
- Id         : unique patient ID
- BIRTHDATE  : date of birth
- DEATHDATE  : date of death (null if alive)
- GENDER     : 'M' or 'F'
- RACE       : e.g. 'white', 'black', 'asian', 'hispanic'
- CITY       : city name
- STATE      : state name
"""

In [0]:
SQL_PROMPT = ChatPromptTemplate.from_messages([
    ("system", f"""You are a Spark SQL expert working on Azure Databricks.
Generate a valid Spark SQL query for the patients table.

{SCHEMA_INFO}

Rules:
- Always use the fully qualified table name: {TABLE}
- Return ONLY the SQL query. No explanation, no markdown, no backticks."""),
    ("human", "{query}")
])

SUMMARY_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful healthcare data assistant. Given a question, SQL query, and result, write a clear one-sentence natural language answer."),
    ("human", """Question: {query}
SQL: {sql}
Result: {result}""")
])

def run_query(question: str):
    # Step 1 — Generate SQL
    sql = (SQL_PROMPT | llm).invoke({"query": question}).content.strip()
    print(f"SQL:\n{sql}\n")

    # Step 2 — Execute
    try:
        result_df = spark.sql(sql)
        result    = result_df.collect()
        print("Result:")
        result_df.show()
    except Exception as e:
        print(f"❌ Execution error: {e}")
        return

    # Step 3 — Summarize
    summary = (SUMMARY_PROMPT | llm).invoke({
        "query":  question,
        "sql":    sql,
        "result": str(result[:5])
    }).content.strip()

    print(f"\n💬 {summary}")

In [0]:
run_query("How many female patients are there who have dibeties?")